In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Setup & imports

In [1]:
# --- Optional: installs for a fresh Colab ---
%pip install camel-tools regex numpy pandas scipy scikit-learn
%pip install -U "transformers==4.36.2" "sentence-transformers==2.5.1" torch

  Using cached transformers-4.36.2-py3-none-any.whl.metadata (126 kB)
  Using cached sentence_transformers-2.5.1-py3-none-any.whl.metadata (11 kB)
  Using cached torch-2.9.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (30 kB)
  Using cached tokenizers-0.15.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.7 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.8.93-py3-none-manylinux2010_x86_64.manylinux_2_12_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cuda_runtime_cu12-12.8.90-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cuda_cupti_cu12-12.8.90-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cublas_cu12-12.8.4.1-py3-none-manylinux_2_27_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cufft_cu12-11.3.3.83-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_curand_cu12-10.3.9.90-py3-none-manylinux_2_27_x86_64.whl.metadat

In [2]:
#(Optional) CAMeL data (small/light)
!camel_data -i light

The following packages will be installed: 'morphology-db-msa-r13', 'disambig-mle-calima-egy-r13', 'morphology-db-lev-01', 'morphology-db-egy-r13', 'dialectid-model26', 'disambig-mle-calima-msa-r13', 'morphology-db-msa-s31', 'morphology-db-glf-01'
Extracting package 'morphology-db-msa-r13': 100% 40.5M/40.5M [00:00<00:00, 213MB/s]
Extracting package 'disambig-mle-calima-egy-r13': 100% 27.2M/27.2M [00:00<00:00, 109MB/s]
Extracting package 'morphology-db-lev-01': 100% 10.6M/10.6M [00:00<00:00, 292MB/s]
Extracting package 'morphology-db-egy-r13': 100% 67.3M/67.3M [00:00<00:00, 274MB/s]
Extracting package 'dialectid-model26': 100% 371M/371M [00:01<00:00, 251MB/s]
Extracting package 'disambig-mle-calima-msa-r13': 100% 88.7M/88.7M [00:00<00:00, 257MB/s]
Extracting package 'morphology-db-msa-s31': 100% 44.8M/44.8M [00:00<00:00, 568MB/s]
Extracting package 'morphology-db-glf-01': 100% 7.98M/7.98M [00:00<00:00, 579MB/s]


In [3]:
import os
import math
import numpy as np
import pandas as pd
import regex as re
from collections import Counter

# Split utilities
from sklearn.model_selection import train_test_split

# CAMeL Tools
from camel_tools.disambig.mle import MLEDisambiguator

# Sentence-level embeddings (feature 88)
try:
    from sentence_transformers import SentenceTransformer
    import torch
    HAS_ST = True
except Exception as e:
    print("[warn] sentence-transformers unavailable:", e)
    HAS_ST = False

pd.set_option("display.max_colwidth", 120)


/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


## Config & I/O

In [4]:
SEED = 42
INPUT_CSV = "/content/drive/MyDrive/MSIS-822-Project/v2/data/02_long-clean_phase_2/arabic_generated_abstracts_long_clean_v2.csv"
OUTPUT_DIR = "/content/drive/MyDrive/MSIS-822-Project/v2/data/03_phase3_features"
os.makedirs(OUTPUT_DIR, exist_ok=True)

## Load & choose text column

In [10]:
df = pd.read_csv(INPUT_CSV)

if "text" in df.columns:
    TEXT_COL = "text"
else:
    raise ValueError("No text column found. Expected 'text_clean' or 'text'.")

RAW_COL = None  # set this to 'text_raw' if you have raw text with punctuation/diacritics

## Split FIRST (70/15/15), no leakage

In [ ]:
# We stratify if a 'label' column exists (0/1 or string). Otherwise, random split.
stratify_col = df["label"] if "label" in df.columns else None

# Train vs temp (70 / 30)
df_train, df_temp = train_test_split(
    df,
    test_size=0.30,
    random_state=SEED,
    stratify=stratify_col
)

# Val vs Test (15 / 15 of total → 50/50 of the temp)
stratify_temp = df_temp["label"] if "label" in df_temp.columns else None
df_val, df_test = train_test_split(
    df_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=stratify_temp
)

# Mark splits
df_train = df_train.copy(); df_train["split"] = "train"
df_val   = df_val.copy();   df_val["split"]   = "val"
df_test  = df_test.copy();  df_test["split"]  = "test"

print(f"Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}")

## Arabic utilities

In [ ]:
AR_SENT_SPLIT = re.compile(r"[\.!\?؟…]+")
WS_SPLIT      = re.compile(r"\s+")
TANWEEN_CHARS = set("\u064B\u064C\u064D")         # ً ٌ ٍ
LINK_RE = re.compile(r"https?://\S+")

# CAMeL Disambiguator (MSA default)
mle = MLEDisambiguator.pretrained()

def ar_sentences(text: str):
    if not isinstance(text, str):
        text = "" if text is None else str(text)
    return [s.strip() for s in AR_SENT_SPLIT.split(text) if s.strip()]

def ar_tokens(text: str):
    if not isinstance(text, str):
        text = "" if text is None else str(text)
    toks = [t for t in WS_SPLIT.split(text) if t]
    # Arabic-first tokens (letters/digits/underscore next)
    return [t for t in toks if re.match(r"^\p{Arabic}[\p{Arabic}\p{Nd}_]*$", t)]

def camel_disamb_tokens(tokens):
    """Return list of dicts per token: word, lemma, pos, vox, cas, num, per, gen."""
    out = []
    if not tokens:
        return out
    disamb = mle.disambiguate(tokens)
    for d in disamb:
        if getattr(d, "analyses", None):
            a = d.analyses[0].analysis
            out.append({
                "word": d.word,
                "lemma": a.get("lemma", d.word),
                "pos": a.get("pos", None),  # 'noun', 'verb', 'adj', 'adv', ...
                "vox": a.get("vox", None),  # 'act', 'pass'
                "cas": a.get("cas", None),  # 'nom', 'acc', 'gen'
                "num": a.get("num", None),  # 'sg', 'du', 'pl'
                "per": a.get("per", None),
                "gen": a.get("gen", None),
            })
        else:
            out.append({"word": d.word, "lemma": d.word, "pos": None, "vox": None, "cas": None,
                        "num": None, "per": None, "gen": None})
    return out

## Feature functions (assigned features only)

In [ ]:
from math import log2

def feat_1_total_chars(text:str)->int:
    return len(text if isinstance(text, str) else str(text))

def feat_4_ws_over_C(text:str)->float:
    text = str(text) if not isinstance(text, str) else text
    C = len(text)
    if C == 0:
        return 0.0
    whites = sum(1 for ch in text if ch.isspace())
    return whites / C

def feat_13_hapax_ratio(tokens)->float:
    if not tokens:
        return 0.0
    freq = Counter(tokens)
    hapax = sum(1 for _, c in freq.items() if c == 1)
    return hapax / len(tokens)

def feat_22_entropy_of_word_freq(tokens)->float:
    if not tokens:
        return 0.0
    freq = Counter(tokens)
    N = sum(freq.values())
    H = 0.0
    for c in freq.values():
        p = c / N
        H -= p * log2(p)
    return float(H)

def feat_25_single_quotes(text:str)->int:
    text = str(text) if not isinstance(text, str) else text
    return text.count("'") + text.count("’") + text.count("‘")

def feat_34_total_sentences(sentences)->int:
    return len(sentences)

def feat_43_num_nouns(analyses)->int:
    return sum(1 for a in analyses if (a.get("pos","") or "").lower().startswith("noun"))

def feat_46_num_adverbs(analyses)->int:
    return sum(1 for a in analyses if (a.get("pos","") or "").lower().startswith("adv"))

def feat_55_noun_to_verb_ratio(analyses)->float:
    n_n = sum(1 for a in analyses if (a.get("pos","") or "").lower().startswith("noun"))
    n_v = sum(1 for a in analyses if (a.get("pos","") or "").lower().startswith("verb"))
    if n_v == 0:
        return float("inf") if n_n > 0 else 0.0
    return n_n / n_v

def feat_64_num_nominatives(analyses)->int:
    # case == 'nom' (CAMeL may be sparse without diacritics)
    return sum(1 for a in analyses if (a.get("cas","") or "").lower().startswith("n"))

def feat_67_num_singular_words(analyses)->int:
    return sum(1 for a in analyses if (a.get("num","") or "").lower().startswith("s"))

def feat_76_num_passive_sentences(sentences, analyses_per_sentence)->int:
    cnt = 0
    for anal in analyses_per_sentence:
        has_pass = any((a.get("vox","") or "").lower().startswith("p") for a in anal)
        if has_pass:
            cnt += 1
    return cnt

def feat_85_sent_len_variance(sentences)->float:
    if not sentences:
        return 0.0
    lengths = [len(ar_tokens(s)) for s in sentences]
    return float(np.var(lengths)) if lengths else 0.0

def feat_106_tanween_freq(text_raw_or_clean:str)->int:
    text_raw_or_clean = str(text_raw_or_clean) if not isinstance(text_raw_or_clean, str) else text_raw_or_clean
    return sum(1 for ch in text_raw_or_clean if ch in TANWEEN_CHARS)

def feat_109_link_freq(text:str)->int:
    text = str(text) if not isinstance(text, str) else text
    return len(LINK_RE.findall(text))


## global knobs

In [ ]:
# ---------- Performance knobs ----------
ENABLE_F88 = True    # set False to skip sentence similarity
ENABLE_F97 = True    # set False to skip BERT token-level similarity

# Caps per-doc to keep work bounded
MAX_SENTS_F88 = 12       # only first N sentences per doc for f088
MAX_CHARS_F97 = 400      # only first N chars per doc for f097 tokenization
MAX_TOKS_F97  = 256      # cap subword tokens for f097

# Chunking & checkpoints
CHUNK_SIZE = 2000        # process docs in chunks
CHECKPOINT_DIR = OUTPUT_DIR
os.makedirs(CHECKPOINT_DIR, exist_ok=True)


## Load Models

In [ ]:
# Device for torch models
_device = None
def _torch_device():
    global _device
    if _device is None:
        _device = "cuda" if torch.cuda.is_available() else "cpu"
    return _device

def feat_88_semantic_sim_sentences(sentences, model=None)->float:
    # Fast exits
    if not ENABLE_F88:
        return float("nan")
    if not sentences or len(sentences) < 2:
        return float("nan")
    if model is None:
        return float("nan")

    # Truncate sentence count
    sents = sentences[:MAX_SENTS_F88]

    # Encode on device
    emb = model.encode(
        sents,
        convert_to_tensor=True,
        normalize_embeddings=True,
        device=_torch_device()
    )
    sims = [float(torch.nn.functional.cosine_similarity(emb[i], emb[i+1], dim=0))
            for i in range(len(emb)-1)]
    return float(np.mean(sims)) if sims else float("nan")


from transformers import AutoTokenizer, AutoModel

_BERT_MODEL = None
_TOKENIZER = None

def _load_bert_token_model():
    """Load Arabic BERT model and tokenizer (lazy) and move to device."""
    global _BERT_MODEL, _TOKENIZER
    if _BERT_MODEL is None or _TOKENIZER is None:
        model_name = "aubmindlab/bert-base-arabertv02"
        _TOKENIZER = AutoTokenizer.from_pretrained(model_name)
        _BERT_MODEL = AutoModel.from_pretrained(model_name).to(_torch_device()).eval()
    return _BERT_MODEL, _TOKENIZER

def feat_97_bert_embedding_similarity_tokens(text: str, token_level_model=None) -> float:
    if not ENABLE_F97:
        return float("nan")

    text = str(text or "").strip()
    if len(text) < 2:
        return float("nan")

    # Truncate the raw string to cap cost
    text = text[:MAX_CHARS_F97]

    try:
        model, tokenizer = _load_bert_token_model() if token_level_model is None else token_level_model
        inputs = tokenizer(
            text, return_tensors="pt",
            truncation=True, max_length=MAX_TOKS_F97
        )
        inputs = {k: v.to(_torch_device()) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs)
            hidden = outputs.last_hidden_state.squeeze(0)  # (seq_len, hidden_size)

        if hidden.size(0) < 3:
            return float("nan")

        sims = torch.nn.functional.cosine_similarity(
            hidden[:-1], hidden[1:], dim=1
        ).detach().cpu().numpy()

        return float(np.mean(sims))
    except Exception as e:
        print(f"[warn] BERT token-sim failed: {e}")
        return float("nan")


## Per-document extraction

In [ ]:
def compute_assigned_features_for_text(text, text_raw=None, st_sentence_model=None):
    # Use provided text for both sentence split and tokenization
    sents = ar_sentences(text)
    toks  = ar_tokens(text)
    anal  = camel_disamb_tokens(toks)

    # Analyses per sentence (for passive detection)
    anal_by_sent = [camel_disamb_tokens(ar_tokens(s)) for s in sents]

    # Student 1
    f1   = feat_1_total_chars(text)
    f22  = feat_22_entropy_of_word_freq(toks)
    f43  = feat_43_num_nouns(anal)
    f64  = feat_64_num_nominatives(anal)
    f85  = feat_85_sent_len_variance(sents)
    f106 = feat_106_tanween_freq(text_raw if text_raw is not None else text)

    # Student 2
    f4   = feat_4_ws_over_C(text)
    f25  = feat_25_single_quotes(text)
    f46  = feat_46_num_adverbs(anal)
    f67  = feat_67_num_singular_words(anal)
    f88  = feat_88_semantic_sim_sentences(sents, model=st_sentence_model)
    f109 = feat_109_link_freq(text)

    # Student 3
    f13  = feat_13_hapax_ratio(toks)
    f34  = feat_34_total_sentences(sents)
    f55  = feat_55_noun_to_verb_ratio(anal)
    f76  = feat_76_num_passive_sentences(sents, anal_by_sent)
    f97  = feat_97_bert_embedding_similarity_tokens(text, token_level_model=None)

    return {
        "f001_total_chars": f1,
        "f004_ws_over_C": f4,
        "f013_hapax_ratio": f13,
        "f022_entropy_wordfreq": f22,
        "f025_single_quotes": f25,
        "f034_total_sentences": f34,
        "f043_num_nouns": f43,
        "f046_num_adverbs": f46,
        "f055_noun_to_verb_ratio": f55,
        "f064_num_nominatives": f64,
        "f067_num_singular": f67,
        "f076_num_passive_sentences": f76,
        "f085_sent_len_variance": f85,
        "f088_sem_sim_sentences": f88,
        "f097_bert_sim_tokens": f97,
        "f106_tanween_freq": f106,
        "f109_link_freq": f109
    }


## Apply features per split & Save results

In [ ]:
from pathlib import Path

def apply_features_dataframe_chunked(
    df_split: pd.DataFrame,
    text_col=TEXT_COL,
    raw_col=RAW_COL,
    model=None,
    chunk_size=CHUNK_SIZE,
    split="train",
    checkpoint_dir=CHECKPOINT_DIR
) -> pd.DataFrame:
    out_rows = []
    total = len(df_split)
    ckpt_path = Path(checkpoint_dir) / f"phase3_{split}_features_ckpt.csv"

    for start in range(0, total, chunk_size):
        end = min(start + chunk_size, total)
        batch = df_split.iloc[start:end]
        rows = []

        for idx, row in batch.iterrows():
            text = row.get(text_col, "")
            text_raw = row.get(raw_col, None) if raw_col else None
            feats = compute_assigned_features_for_text(text, text_raw=text_raw, st_sentence_model=model)
            rows.append(feats)

        feat_df = pd.DataFrame(rows, index=batch.index)
        out_rows.append(pd.concat([batch, feat_df], axis=1))

        # checkpoint this chunk
        pd.concat(out_rows, axis=0).to_csv(ckpt_path, index=False)
        print(f"[{split}] processed {end}/{total}")

    result = pd.concat(out_rows, axis=0)
    # remove checkpoint when done
    try:
        Path(ckpt_path).unlink()
    except Exception:
        pass
    return result


# Load sentence model only once (for f088)
st_model = None
if HAS_ST and ENABLE_F88:
    try:
        st_model = SentenceTransformer("Omartificial-Intelligence-Space/Arabic-arabert-all-nli-triplet")
        # pre-warm a tiny encode on device
        _ = st_model.encode(["اختبار"], convert_to_tensor=True, device=_torch_device())
    except Exception as e:
        print("[warn] could not load sentence model:", e)
        st_model = None

# Run with chunking and save final CSVs
train_feats = apply_features_dataframe_chunked(df_train, text_col=TEXT_COL, raw_col=RAW_COL,
                                               model=st_model, split="train")
val_feats   = apply_features_dataframe_chunked(df_val,   text_col=TEXT_COL, raw_col=RAW_COL,
                                               model=st_model, split="val")
test_feats  = apply_features_dataframe_chunked(df_test,  text_col=TEXT_COL, raw_col=RAW_COL,
                                               model=st_model, split="test")

train_csv = os.path.join(OUTPUT_DIR, "phase3_train_features.csv")
val_csv   = os.path.join(OUTPUT_DIR, "phase3_val_features.csv")
test_csv  = os.path.join(OUTPUT_DIR, "phase3_test_features.csv")

train_feats.to_csv(train_csv, index=False)
val_feats.to_csv(val_csv, index=False)
test_feats.to_csv(test_csv, index=False)

print("Saved:")
print(train_csv)
print(val_csv)
print(test_csv)

all_feats = pd.concat([train_feats, val_feats, test_feats], axis=0, ignore_index=True)
all_csv = os.path.join(OUTPUT_DIR, "phase3_all_features_with_split.csv")
all_feats.to_csv(all_csv, index=False)
print("All-in-one CSV:", all_csv)


In [ ]:
# Optional peek
all_feats.head(10)